# External Method PK for SOXL Overfit Bottleneck v1

This notebook ranks external-method ideas against the current SOXL bottleneck:

- target excess return: `>= +30pp`
- target MaxDD: `<= 15%`
- avoid overfit through walk-forward validation and notebook-first evidence
- keep reusable logic in `src/`, experiment-specific logic in `runtime/`

## Current Bottleneck

The latest reward screens show that `short_volshock_plain` can learn the short-volshock de-risk labels, but official walk-forward replay still fails:

- `wf_2022`: drawdown control failure, MaxDD above 15%.
- `wf_2023_2024`: joint tradeoff, relative return far below +30pp because trend participation is too low.

So the PK question is not "what is the most advanced model?" but "which method class is most likely to solve drawdown drift and trend underparticipation without overfitting?"

## External Sources Used

- Qlib / arXiv: AI-oriented quant workflow and infrastructure; Qlib supports supervised learning, market dynamics modeling, and RL.
- FinRL GitHub / arXiv: train-test-trade style financial RL and production-oriented modularity.
- Constrained Policy Optimization: separate reward and constraints instead of hiding safety/risk in one scalar reward.
- Temporal Fusion Transformer: interpretable multi-horizon architecture with gating and feature selection.
- Decision Transformer: offline RL as sequence modeling conditioned on target return.
- Twitter/X: considered only as an idea stream; no stable citable X source is used as evidence in this notebook.


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path("/Users/rziiiii/.openclaw/workspace")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Markdown, display

EXPERIMENT = "external_method_pk_overfit_bottleneck_v1"
CARD_DIR = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
CARD_DIR.mkdir(parents=True, exist_ok=True)

REWARD_RECIPE_SUMMARY = ROOT / "runtime" / "reward" / "card" / "short_volshock_drawdown_drift_reward_recipe_screen_v1" / "summary.json"
REWARD_CHECKPOINT_SUMMARY = ROOT / "runtime" / "reward" / "card" / "short_volshock_reward_checkpoint_screen_v1" / "summary.json"
DRAW_AUDIT_SUMMARY = ROOT / "runtime" / "model_selection" / "card" / "short_volshock_drawdown_drift_feature_audit_v1" / "summary.json"

reward_recipe = json.loads(REWARD_RECIPE_SUMMARY.read_text(encoding="utf-8"))
reward_checkpoint = json.loads(REWARD_CHECKPOINT_SUMMARY.read_text(encoding="utf-8"))
draw_audit = json.loads(DRAW_AUDIT_SUMMARY.read_text(encoding="utf-8"))

print(f"card_dir={CARD_DIR.relative_to(ROOT)}")
print(f"latest_reward_recipe_decision={reward_recipe['decision']}")


card_dir=runtime/model_selection/card/external_method_pk_overfit_bottleneck_v1
latest_reward_recipe_decision=drawdown_drift_reward_recipe_ready_for_short_training_screen


## Candidate Method Pool

Scores are intentionally simple and explicit:

- `bottleneck_fit`: can it directly address drawdown drift + trend underparticipation?
- `data_readiness`: can we evaluate it now with existing SOXL/SOXX/VIX/daily-path artifacts?
- `overfit_control`: does it naturally support walk-forward, constraints, or interpretable gates?
- `engineering_cost`: lower is better; converted to `cost_score = 6 - engineering_cost`.
- `external_strength`: how strong and directly applicable the external basis is.


In [2]:
candidates = [
    {
        "method_id": "reward_constrained_short_training_screen",
        "category": "reward",
        "idea": "Train a short fold-local BC/PPO candidate with explicit drawdown-drift penalty plus trend-preservation floor.",
        "external_basis": "CPO-style reward + constraint separation; FinRL train-test-trade loop; Qlib RL workflow.",
        "repo_basis": "reward recipe screen selected dd1.5_thr15_pos0.05_tf1_floor0.25_lb63 with 87.4% drawdown capture and 100% trend capture.",
        "first_experiment": "runtime/reward/card/<short_training_screen>.ipynb using reward_objective_patch.json, then official wf_2022/wf_2023_2024 replay.",
        "expected_gain": "Directly tests whether the current objective bottleneck can be trained out.",
        "failure_mode": "Reward pressure may still drift into higher MaxDD or suppress trend after policy optimization.",
        "bottleneck_fit": 5,
        "data_readiness": 5,
        "overfit_control": 4,
        "engineering_cost": 3,
        "external_strength": 5,
    },
    {
        "method_id": "constrained_checkpoint_selector",
        "category": "model_selection",
        "idea": "Rank checkpoints by a constrained objective: pass MaxDD first, then relative return, then drift vs reference.",
        "external_basis": "Constrained RL frames risk as constraints; Qlib model rolling/checkpoint workflow supports selection gates.",
        "repo_basis": "Current selection by validation/BC alignment did not enforce both walk-forward windows hard enough.",
        "first_experiment": "runtime/model_selection/card/<constrained_checkpoint_selector>.ipynb over existing checkpoint_objective_scores + new short-training checkpoints.",
        "expected_gain": "Reduces overfit by making promotion depend on constraint satisfaction instead of best reward/validation score.",
        "failure_mode": "If no candidate learns the joint behavior, selector only confirms no-promotion.",
        "bottleneck_fit": 5,
        "data_readiness": 4,
        "overfit_control": 5,
        "engineering_cost": 2,
        "external_strength": 4,
    },
    {
        "method_id": "single_factor_soxx_soxl_spread_ic",
        "category": "single_factor",
        "idea": "Mine SOXX/SOXL spread widening, SOXL/SOXX relative oscillation, VIX percentile and trend-quality as single signals before using them in RL.",
        "external_basis": "Qlib factor workflow; repo already has SOXL/SOXX spread feature family.",
        "repo_basis": "Feature audit top groups include trend_quality, volatility_regime, relative_strength_spread; user previously asked about SOXX/SOXL and SOXL/SOXX spread widening.",
        "first_experiment": "runtime/factor/card/<spread_ic_screen>.ipynb with IC, signal accuracy, single-signal WF gate.",
        "expected_gain": "Separates genuine predictive signal from reward/model overfit before expanding observations.",
        "failure_mode": "Signal may be useful only as execution/risk state, not standalone alpha factor.",
        "bottleneck_fit": 4,
        "data_readiness": 5,
        "overfit_control": 5,
        "engineering_cost": 2,
        "external_strength": 4,
    },
    {
        "method_id": "multi_factor_state_guard",
        "category": "multi_factor",
        "idea": "Build a compact state guard from trend_quality + volatility_regime + drawdown_path + relative_strength_spread, then use it only in training/selection gates.",
        "external_basis": "Qlib feature/factor mining workflow and interpretable regime/state models.",
        "repo_basis": "Drawdown drift audit found shared poor-window clusters and stable top feature groups across candidates.",
        "first_experiment": "runtime/model_selection/card/<state_guard_screen>.ipynb with fold-local thresholds and no runtime promotion.",
        "expected_gain": "Better state representation for when to deleverage and when to preserve trend participation.",
        "failure_mode": "Combining many audited features can overfit if thresholds are not fold-local and sparse.",
        "bottleneck_fit": 5,
        "data_readiness": 4,
        "overfit_control": 4,
        "engineering_cost": 3,
        "external_strength": 4,
    },
    {
        "method_id": "tft_or_gated_attention_feature_selector",
        "category": "network",
        "idea": "Use TFT-style gating/attention only as a feature/state selector or small head, not a full large model replacement.",
        "external_basis": "Temporal Fusion Transformer uses gating and variable selection for interpretable multi-horizon forecasting.",
        "repo_basis": "Current bottleneck needs state-dependent behavior; existing policy models include attention/GRU/MOE surfaces.",
        "first_experiment": "runtime/network/card/<gated_feature_selector>.ipynb with tiny capacity and walk-forward ablation vs MLP/GRU.",
        "expected_gain": "May improve regime/state recognition without blind feature expansion.",
        "failure_mode": "High risk of fitting 2022/2024 quirks if used as a large end-to-end actor.",
        "bottleneck_fit": 3,
        "data_readiness": 3,
        "overfit_control": 3,
        "engineering_cost": 4,
        "external_strength": 4,
    },
    {
        "method_id": "decision_transformer_offline_bc",
        "category": "model",
        "idea": "Treat trajectories as offline sequence modeling conditioned on target return/drawdown, then replay as a BC-style candidate.",
        "external_basis": "Decision Transformer casts RL as conditional sequence modeling with desired return conditioning.",
        "repo_basis": "BC target generation exists, but current warm-start replay failed; sequence conditioning might express target tradeoffs better.",
        "first_experiment": "runtime/model_selection/card/<decision_transformer_bc_probe>.ipynb on frozen trajectories and small context windows.",
        "expected_gain": "Could represent path-dependent tradeoffs more directly than one-step BC.",
        "failure_mode": "Large model/data mismatch; high overfit risk for single ticker and limited walk-forward folds.",
        "bottleneck_fit": 3,
        "data_readiness": 3,
        "overfit_control": 2,
        "engineering_cost": 5,
        "external_strength": 4,
    },
    {
        "method_id": "tdqn_or_q_learning_action_value_probe",
        "category": "model",
        "idea": "Run a tiny action-value probe on discrete position buckets to see if policy-gradient PPO is the problem.",
        "external_basis": "Trading DQN literature adapts DQN to trading position decisions and stresses objective performance assessment.",
        "repo_basis": "Current policy path is PPO/BC-heavy; a discrete value probe can diagnose action preference without replacing runtime.",
        "first_experiment": "runtime/model_selection/card/<discrete_value_probe>.ipynb with fixed buckets and purged walk-forward.",
        "expected_gain": "Diagnostic separation between representation/reward issue and PPO optimization issue.",
        "failure_mode": "Coarse action buckets may not map cleanly to current execution and risk budget semantics.",
        "bottleneck_fit": 2,
        "data_readiness": 3,
        "overfit_control": 3,
        "engineering_cost": 4,
        "external_strength": 3,
    },
    {
        "method_id": "twitter_x_idea_watchlist_only",
        "category": "external_watchlist",
        "idea": "Use Twitter/X quant discussions only as an idea watchlist, never as promotion evidence.",
        "external_basis": "Social source, not stable enough for repo evidence without paper/code/data follow-up.",
        "repo_basis": "AGENTS/SPEC require durable artifacts and proof; chat/social snippets are not sources of truth.",
        "first_experiment": "None until an idea is backed by a paper, GitHub project, dataset, or reproducible feature definition.",
        "expected_gain": "Can suggest search terms, but should not steer immediate training.",
        "failure_mode": "Narrative chasing and overfit if social ideas bypass IC/WF gates.",
        "bottleneck_fit": 1,
        "data_readiness": 1,
        "overfit_control": 1,
        "engineering_cost": 1,
        "external_strength": 1,
    },
]

methods = pd.DataFrame(candidates)
methods


,method_id,category,idea,external_basis,repo_basis,first_experiment,expected_gain,failure_mode,bottleneck_fit,data_readiness,overfit_control,engineering_cost,external_strength
0,reward_constrained_short_training_screen,reward,Train a short fold-local BC/PPO candidate with...,CPO-style reward + constraint separation; FinR...,reward recipe screen selected dd1.5_thr15_pos0...,runtime/reward/card/<short_training_screen>.ip...,Directly tests whether the current objective b...,Reward pressure may still drift into higher Ma...,5,5,4,3,5
1,constrained_checkpoint_selector,model_selection,Rank checkpoints by a constrained objective: p...,Constrained RL frames risk as constraints; Qli...,Current selection by validation/BC alignment d...,runtime/model_selection/card/<constrained_chec...,Reduces overfit by making promotion depend on ...,"If no candidate learns the joint behavior, sel...",5,4,5,2,4
2,single_factor_soxx_soxl_spread_ic,single_factor,"Mine SOXX/SOXL spread widening, SOXL/SOXX rela...",Qlib factor workflow; repo already has SOXL/SO...,Feature audit top groups include trend_quality...,runtime/factor/card/<spread_ic_screen>.ipynb w...,Separates genuine predictive signal from rewar...,Signal may be useful only as execution/risk st...,4,5,5,2,4
3,multi_factor_state_guard,multi_factor,Build a compact state guard from trend_quality...,Qlib feature/factor mining workflow and interp...,Drawdown drift audit found shared poor-window ...,runtime/model_selection/card/<state_guard_scre...,Better state representation for when to deleve...,Combining many audited features can overfit if...,5,4,4,3,4
4,tft_or_gated_attention_feature_selector,network,Use TFT-style gating/attention only as a featu...,Temporal Fusion Transformer uses gating and va...,Current bottleneck needs state-dependent behav...,runtime/network/card/<gated_feature_selector>....,May improve regime/state recognition without b...,High risk of fitting 2022/2024 quirks if used ...,3,3,3,4,4
5,decision_transformer_offline_bc,model,Treat trajectories as offline sequence modelin...,Decision Transformer casts RL as conditional s...,"BC target generation exists, but current warm-...",runtime/model_selection/card/<decision_transfo...,Could represent path-dependent tradeoffs more ...,Large model/data mismatch; high overfit risk f...,3,3,2,5,4
6,tdqn_or_q_learning_action_value_probe,model,Run a tiny action-value probe on discrete posi...,Trading DQN literature adapts DQN to trading p...,Current policy path is PPO/BC-heavy; a discret...,runtime/model_selection/card/<discrete_value_p...,Diagnostic separation between representation/r...,Coarse action buckets may not map cleanly to c...,2,3,3,4,3
7,twitter_x_idea_watchlist_only,external_watchlist,Use Twitter/X quant discussions only as an ide...,"Social source, not stable enough for repo evid...",AGENTS/SPEC require durable artifacts and proo...,"None until an idea is backed by a paper, GitHu...","Can suggest search terms, but should not steer...",Narrative chasing and overfit if social ideas ...,1,1,1,1,1


In [3]:
weights = {
    "bottleneck_fit": 0.32,
    "data_readiness": 0.22,
    "overfit_control": 0.24,
    "cost_score": 0.10,
    "external_strength": 0.12,
}
scored = methods.copy()
scored["cost_score"] = 6 - scored["engineering_cost"]
scored["pk_score"] = sum(scored[col] * weight for col, weight in weights.items())
scored = scored.sort_values(["pk_score", "bottleneck_fit", "overfit_control"], ascending=[False, False, False]).reset_index(drop=True)
scored[[
    "method_id",
    "category",
    "pk_score",
    "bottleneck_fit",
    "data_readiness",
    "overfit_control",
    "engineering_cost",
    "external_strength",
    "first_experiment",
]]


,method_id,category,pk_score,bottleneck_fit,data_readiness,overfit_control,engineering_cost,external_strength,first_experiment
0,constrained_checkpoint_selector,model_selection,4.56,5,4,5,2,4,runtime/model_selection/card/<constrained_chec...
1,reward_constrained_short_training_screen,reward,4.56,5,5,4,3,5,runtime/reward/card/<short_training_screen>.ip...
2,single_factor_soxx_soxl_spread_ic,single_factor,4.46,4,5,5,2,4,runtime/factor/card/<spread_ic_screen>.ipynb w...
3,multi_factor_state_guard,multi_factor,4.22,5,4,4,3,4,runtime/model_selection/card/<state_guard_scre...
4,tft_or_gated_attention_feature_selector,network,3.02,3,3,3,4,4,runtime/network/card/<gated_feature_selector>....
5,decision_transformer_offline_bc,model,2.68,3,3,2,5,4,runtime/model_selection/card/<decision_transfo...
6,tdqn_or_q_learning_action_value_probe,model,2.58,2,3,3,4,3,runtime/model_selection/card/<discrete_value_p...
7,twitter_x_idea_watchlist_only,external_watchlist,1.40,1,1,1,1,1,"None until an idea is backed by a paper, GitHu..."


## PK Result

Interpretation rules:

1. Reward/checkpoint routes win only if they can be verified by short walk-forward replay.
2. Factor routes must pass IC/single-signal/WF gates before entering observations or runtime logic.
3. Network/model routes need a small-capacity diagnostic first; no large end-to-end actor until the reward/constraint signal is proven.


In [4]:
top_methods = scored.head(4).copy()
selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": "method_pk_next_batch_selected",
    "selected_methods": top_methods["method_id"].tolist(),
    "primary_next": str(top_methods.iloc[0]["method_id"]),
    "secondary_next": str(top_methods.iloc[1]["method_id"]),
    "parallel_research_next": str(top_methods.iloc[2]["method_id"]),
    "do_not_do_next": [
        "large_transformer_actor_before_reward_constraint_screen",
        "runtime_overlay_promotion_from_objective_screen",
        "twitter_x_idea_without_paper_code_data_gate",
    ],
    "source_context": {
        "reward_recipe_decision": reward_recipe.get("decision"),
        "reward_checkpoint_decision": reward_checkpoint.get("decision"),
        "drawdown_drift_audit_decision": draw_audit.get("decision"),
    },
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://github.com/AI4Finance-Foundation/FinRL",
        "https://arxiv.org/abs/2011.09607",
        "https://arxiv.org/abs/1705.10528",
        "https://arxiv.org/abs/1912.09363",
        "https://arxiv.org/abs/2106.01345",
        "https://arxiv.org/abs/2004.06627",
    ],
    "sidecars": ["method_candidates.csv", "method_pk_scores.csv", "selection.json", "summary.json", "method_pk_review.md"],
}
summary = {
    **selection,
    "conclusion": [
        "Primary next: run the constrained reward short-training screen because it directly targets the confirmed bottleneck and is data-ready.",
        "Second: build a constrained checkpoint selector so promotion is constraint-first, not validation-score-first.",
        "Parallel low-risk research: screen SOXX/SOXL spread and related volatility/trend features as single factors before feeding them into RL.",
        "Network/model upgrades such as TFT-style gating or Decision Transformer are deferred until the reward/constraint signal is proven, because they carry higher overfit and engineering risk.",
    ],
}
summary


{'experiment': 'external_method_pk_overfit_bottleneck_v1',
 'created_at': '2026-05-26T14:20:22.354248+00:00',
 'decision': 'method_pk_next_batch_selected',
 'selected_methods': ['constrained_checkpoint_selector',
  'reward_constrained_short_training_screen',
  'single_factor_soxx_soxl_spread_ic',
  'multi_factor_state_guard'],
 'primary_next': 'constrained_checkpoint_selector',
 'secondary_next': 'reward_constrained_short_training_screen',
 'parallel_research_next': 'single_factor_soxx_soxl_spread_ic',
 'do_not_do_next': ['large_transformer_actor_before_reward_constraint_screen',
  'runtime_overlay_promotion_from_objective_screen',
  'twitter_x_idea_without_paper_code_data_gate'],
 'source_context': {'reward_recipe_decision': 'drawdown_drift_reward_recipe_ready_for_short_training_screen',
  'reward_checkpoint_decision': 'reward_checkpoint_screen_no_promotion',
  'drawdown_drift_audit_decision': 'short_volshock_drawdown_drift_joint_objective_needed_no_promotion'},
 'external_sources': [

In [5]:
def render_review(scored: pd.DataFrame, summary: dict) -> str:
    lines = [
        "# External Method PK",
        "",
        "## Decision",
        "",
        f"- decision: `{summary['decision']}`",
        f"- primary_next: `{summary['primary_next']}`",
        f"- secondary_next: `{summary['secondary_next']}`",
        f"- parallel_research_next: `{summary['parallel_research_next']}`",
        "",
        "## Ranking",
        "",
        "| rank | method | category | score | fit | data | overfit ctrl | cost | first experiment |",
        "| ---: | --- | --- | ---: | ---: | ---: | ---: | ---: | --- |",
    ]
    for idx, row in scored.iterrows():
        lines.append(
            f"| {idx + 1} | {row['method_id']} | {row['category']} | {row['pk_score']:.2f} | "
            f"{row['bottleneck_fit']} | {row['data_readiness']} | {row['overfit_control']} | "
            f"{row['engineering_cost']} | {row['first_experiment']} |"
        )
    lines.extend(["", "## Conclusion", ""])
    for item in summary["conclusion"]:
        lines.append(f"- {item}")
    lines.extend(["", "## Do Not Do Next", ""])
    for item in summary["do_not_do_next"]:
        lines.append(f"- `{item}`")
    return "\n".join(lines).rstrip() + "\n"

review_md = render_review(scored, summary)
methods.to_csv(CARD_DIR / "method_candidates.csv", index=False)
scored.to_csv(CARD_DIR / "method_pk_scores.csv", index=False)
(CARD_DIR / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD_DIR / "method_pk_review.md").write_text(review_md, encoding="utf-8")
print(review_md)

# External Method PK

## Decision

- decision: `method_pk_next_batch_selected`
- primary_next: `constrained_checkpoint_selector`
- secondary_next: `reward_constrained_short_training_screen`
- parallel_research_next: `single_factor_soxx_soxl_spread_ic`

## Ranking

| rank | method | category | score | fit | data | overfit ctrl | cost | first experiment |
| ---: | --- | --- | ---: | ---: | ---: | ---: | ---: | --- |
| 1 | constrained_checkpoint_selector | model_selection | 4.56 | 5 | 4 | 5 | 2 | runtime/model_selection/card/<constrained_checkpoint_selector>.ipynb over existing checkpoint_objective_scores + new short-training checkpoints. |
| 2 | reward_constrained_short_training_screen | reward | 4.56 | 5 | 5 | 4 | 3 | runtime/reward/card/<short_training_screen>.ipynb using reward_objective_patch.json, then official wf_2022/wf_2023_2024 replay. |
| 3 | single_factor_soxx_soxl_spread_ic | single_factor | 4.46 | 4 | 5 | 5 | 2 | runtime/factor/card/<spread_ic_screen>.ipynb with IC, signal a

In [6]:
display(Markdown(review_md))


# External Method PK

## Decision

- decision: `method_pk_next_batch_selected`
- primary_next: `constrained_checkpoint_selector`
- secondary_next: `reward_constrained_short_training_screen`
- parallel_research_next: `single_factor_soxx_soxl_spread_ic`

## Ranking

| rank | method | category | score | fit | data | overfit ctrl | cost | first experiment |
| ---: | --- | --- | ---: | ---: | ---: | ---: | ---: | --- |
| 1 | constrained_checkpoint_selector | model_selection | 4.56 | 5 | 4 | 5 | 2 | runtime/model_selection/card/<constrained_checkpoint_selector>.ipynb over existing checkpoint_objective_scores + new short-training checkpoints. |
| 2 | reward_constrained_short_training_screen | reward | 4.56 | 5 | 5 | 4 | 3 | runtime/reward/card/<short_training_screen>.ipynb using reward_objective_patch.json, then official wf_2022/wf_2023_2024 replay. |
| 3 | single_factor_soxx_soxl_spread_ic | single_factor | 4.46 | 4 | 5 | 5 | 2 | runtime/factor/card/<spread_ic_screen>.ipynb with IC, signal accuracy, single-signal WF gate. |
| 4 | multi_factor_state_guard | multi_factor | 4.22 | 5 | 4 | 4 | 3 | runtime/model_selection/card/<state_guard_screen>.ipynb with fold-local thresholds and no runtime promotion. |
| 5 | tft_or_gated_attention_feature_selector | network | 3.02 | 3 | 3 | 3 | 4 | runtime/network/card/<gated_feature_selector>.ipynb with tiny capacity and walk-forward ablation vs MLP/GRU. |
| 6 | decision_transformer_offline_bc | model | 2.68 | 3 | 3 | 2 | 5 | runtime/model_selection/card/<decision_transformer_bc_probe>.ipynb on frozen trajectories and small context windows. |
| 7 | tdqn_or_q_learning_action_value_probe | model | 2.58 | 2 | 3 | 3 | 4 | runtime/model_selection/card/<discrete_value_probe>.ipynb with fixed buckets and purged walk-forward. |
| 8 | twitter_x_idea_watchlist_only | external_watchlist | 1.40 | 1 | 1 | 1 | 1 | None until an idea is backed by a paper, GitHub project, dataset, or reproducible feature definition. |

## Conclusion

- Primary next: run the constrained reward short-training screen because it directly targets the confirmed bottleneck and is data-ready.
- Second: build a constrained checkpoint selector so promotion is constraint-first, not validation-score-first.
- Parallel low-risk research: screen SOXX/SOXL spread and related volatility/trend features as single factors before feeding them into RL.
- Network/model upgrades such as TFT-style gating or Decision Transformer are deferred until the reward/constraint signal is proven, because they carry higher overfit and engineering risk.

## Do Not Do Next

- `large_transformer_actor_before_reward_constraint_screen`
- `runtime_overlay_promotion_from_objective_screen`
- `twitter_x_idea_without_paper_code_data_gate`


## Final Route

Recommended next batch:

1. **Primary**: constrained reward short-training screen using the selected drawdown-drift + trend-preservation patch.
2. **Secondary**: constrained checkpoint selector, so any trained candidate must satisfy MaxDD first and relative return second.
3. **Parallel research**: single-factor SOXX/SOXL spread / VIX / trend-quality IC and single-signal WF screen.

Deferred:

- large network change before reward/constraint signal is proven;
- Decision Transformer or DQN as production replacement;
- any Twitter/X idea without a paper, GitHub project, dataset, or reproducible factor definition.
